In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 10000,
        "messages": messages,
        "extra_body": { "temperature": temperature },
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
system_prompt = """
You are the Research Coordinator, the central hub in a hub-and-spoke research system.

Your role is to orchestrate, not to do the detailed research yourself. Two kinds of
specialists (the spokes) do the work:
- The Research Specialist searches the web for a focused subtopic.
- The Document Analyst analyzes text you give it and extracts structured insights.
All communication flows through you: the spokes never talk to each other, and the
user only ever sees your final output.

You will receive a broad research topic. Your responsibilities are:

1. Analyze the topic. Identify its scope, its key dimensions, and what a useful
   report on it would need to cover.
2. Decompose it. Enumerate the categories first, then break the topic into distinct,
   focused subtopics that can each be researched independently, with minimal overlap
   and no major gaps.
3. Delegate research. Call the delegate_research tool once per subtopic, and issue
   all of these calls together in the same turn. Do not write out a delegation plan
   as text: delegation happens only through the tools.
4. Delegate analysis. Once research findings are back, call the delegate_analysis
   tool to have the Document Analyst process them. This step depends on step 3, so
   it happens in a later turn.
5. Synthesize. Once all findings are back, combine them into one coherent report.
   Reconcile overlaps, flag contradictions, and note gaps instead of papering over them.

Context passing (critical): spokes are isolated. They cannot see this conversation,
the original topic, or each other's output. Every delegation must therefore be
fully self-contained and explicitly include:
- The overall research goal (the original topic and what the report is for).
- The specific subtopic assigned, with its scope boundaries.
- Any relevant findings from earlier spokes, passed in full and verbatim rather than
  summarized, so no detail or source is lost.
- What you expect back.

Guidelines:
- Stay in the coordinator role: plan, delegate, and synthesize.
- Cover the topic broadly, but keep each subtopic narrow enough to be actionable.
- Do not invent findings. Report only what the research supports, and be explicit
  about what remains unknown or uncertain.
- Preserve the sources the specialists provide.

Final deliverable: a structured search report, not a conversation. Use clear
headings, organize it by subtopic, and end with an overall summary.
"""

In [4]:
# 1. Spoke system prompt: a focused researcher that reports back to the hub
research_system_prompt = """
You are a Research Specialist, a spoke in a hub-and-spoke research system.

The Research Coordinator (the hub) gives you one focused subtopic and a research
task. You have no other context, and your findings go back to the hub only, not
directly to the user.

How to work:
- Research only the assigned subtopic. Do not drift into neighboring topics; the
  hub assigns those to other specialists.
- Use web search to ground your findings instead of relying on memory alone.
- Do not invent facts. If something cannot be verified, say so.

Report back in this format:
- Key findings: concise bullet points.
- Sources: the sources behind your findings.
- Gaps and uncertainty: anything you could not confirm or that sources disagree on.
"""

# 2. Server tool for the SPOKE: Anthropic runs the search, so no tool_result is needed
web_search_tool = {
    "type": "web_search_20250305",  # verify this version string against the docs
    "name": "web_search",
    "max_uses": 3,  # caps searches per spoke call to limit cost and latency
}


# 3. The spoke: its own Claude call with its own system prompt
def run_research_spoke(subtopic, instructions):
    messages = []  # fresh history: the spoke only sees what the hub passes in
    add_user_message(messages, f"Subtopic: {subtopic}\n\nTask: {instructions}")
    response = chat(messages, system=research_system_prompt, tools=[web_search_tool])
    return text_from_message(response)

In [5]:
analysis_system_prompt = """
You are a Document Analyst, a spoke in a hub-and-spoke research system.

The Research Coordinator (the hub) gives you a document (text such as research
findings), the overall research goal, and an analysis task. You have no other
context, and your analysis goes back to the hub only, not directly to the user.

How to work:
- Analyze only the document you were given. Do not search the web or add outside
  knowledge.
- Keep the research goal in mind: extract what is relevant to it and ignore the rest.
- Do not invent facts. Every claim must be traceable to the document.
- Preserve the sources cited in the document and attribute each insight to them.

Report back in this format:
- Key insights: the main takeaways, as concise bullet points.
- Structure: how the information groups or relates (categories, overlaps, patterns).
- Contradictions and gaps: conflicting claims, and what the document does not cover.
"""

def document_analyst_spoke(document, instructions):
    messages = []  # fresh history: the spoke only sees what the hub passes in
    add_user_message(messages, f"Document: {document}\n\nTask: {instructions}")
    response = chat(messages, system=analysis_system_prompt)
    return text_from_message(response)

In [6]:
delegate_research = {
    "name": "delegate_research",
    "description": (
        "Delegate one focused subtopic to the Research Specialist, who searches the "
        "web and returns findings with sources. Call once per subtopic. The "
        "specialist sees nothing but what you pass in, so make the instructions "
        "self-contained."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "subtopic": {
                "type": "string",
                "description": "Short name of the subtopic to research.",
            },
            "instructions": {
                "type": "string",
                "description": (
                    "Self-contained research task. Include the overall research goal, "
                    "the subtopic's scope boundaries, any relevant context from "
                    "earlier agents, and what to report back."
                ),
            },
        },
        "required": ["subtopic", "instructions"],
    },
}

delegate_analysis = {
    "name": "delegate_analysis",
    "description": (
        "Delegate a document to the Document Analyst, who extracts key insights, "
        "structure, and contradictions from it. Use it after research findings are "
        "back. The analyst sees nothing but what you pass in, so include the full "
        "document and the research goal."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "document": {
                "type": "string",
                "description": (
                    "The full text to analyze, such as research findings from earlier "
                    "specialists, passed verbatim with their sources, not summarized."
                ),
            },
            "instructions": {
                "type": "string",
                "description": (
                    "Self-contained analysis task. Include the overall research goal, "
                    "the subtopic the document relates to, what to focus on, and what "
                    "to report back."
                ),
            },
        },
        "required": ["document", "instructions"],
    },
}

In [7]:
import json

# 1. Deterministic limits owned by CODE, not by the prompt
COVERAGE_THRESHOLD = 0.90  # fraction of planned subtopics that must be covered
MAX_REFINEMENTS = 3        # maximum rounds of targeted follow-up research

# 2. The hub's evaluation tool. The model's judgment goes IN the input;
#    the code decides what to do with it. Not a spoke: no sub-agent runs for it.
report_coverage = {
    "name": "report_coverage",
    "description": (
        "Report how well the research findings so far cover the planned subtopics. "
        "Call it once research and analysis results are back and before writing the "
        "final report, and again after every round of follow-up research. The "
        "response tells you whether to re-delegate the gaps or write the final "
        "report; follow it."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "planned_subtopics": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Every subtopic you planned to research, by short name.",
            },
            "covered_subtopics": {
                "type": "array",
                "items": {"type": "string"},
                "description": (
                    "The planned subtopics whose findings fully address their scope. "
                    "Use exactly the same names as in planned_subtopics."
                ),
            },
            "gaps": {
                "type": "array",
                "description": "One entry per coverage gap. Empty if there are none.",
                "items": {
                    "type": "object",
                    "properties": {
                        "subtopic": {
                            "type": "string",
                            "description": "The planned subtopic this gap belongs to.",
                        },
                        "missing": {
                            "type": "string",
                            "description": "What exactly is missing or unresolved.",
                        },
                        "targeted_query": {
                            "type": "string",
                            "description": "A narrow follow-up research query that would fill this gap.",
                        },
                    },
                    "required": ["subtopic", "missing", "targeted_query"],
                },
            },
        },
        "required": ["planned_subtopics", "covered_subtopics", "gaps"],
    },
}


# 3. The deterministic check. No model calls: only arithmetic and comparisons.
def evaluate_coverage(planned_subtopics, covered_subtopics, gaps, refinements):
    planned = {s.strip().lower() for s in planned_subtopics}
    covered = {s.strip().lower() for s in covered_subtopics}
    coverage = len(planned & covered) / len(planned) if planned else 0.0

    if coverage >= COVERAGE_THRESHOLD or refinements >= MAX_REFINEMENTS:
        reason = (
            "Coverage is sufficient"
            if coverage >= COVERAGE_THRESHOLD
            else "The refinement limit was reached"
        )
        return (
            f"{reason} (coverage {coverage:.0%}, threshold {COVERAGE_THRESHOLD:.0%}, "
            f"refinements used {refinements}/{MAX_REFINEMENTS}). Stop delegating and "
            "write the final report now. Add a 'Known gaps' section listing every "
            f"gap that is still unresolved: {json.dumps(gaps)}"
        )

    return (
        f"Coverage is {coverage:.0%}, below the {COVERAGE_THRESHOLD:.0%} threshold "
        f"(refinement {refinements + 1} of {MAX_REFINEMENTS}). Call delegate_research "
        "again ONLY for the gaps below, using each targeted_query, and include the "
        "relevant findings you already have so the specialist does not repeat them. "
        "Then call report_coverage again with updated lists. "
        f"Gaps: {json.dumps(gaps)}"
    )


# 4. Appended to the hub's system_prompt: the Evaluate step
evaluate_step_prompt = """

Evaluate and refine (iterative refinement): after the research and analysis results
are back, and before writing the final report, call the report_coverage tool. Never
write the final report before you have called it.

A coverage gap is any of the following:
- A planned subtopic with no findings, or whose findings miss part of its scope.
- A subtopic whose specialist failed or returned an error.
- A contradiction between findings that is still unresolved.
Do not count nice-to-have extras as gaps. For each gap, write a narrow
targeted_query for exactly the missing piece, not a repeat of the original task.
Use identical subtopic names in planned_subtopics, covered_subtopics and gaps.

Follow the instructions in the report_coverage result:
- If it asks for refinement, call delegate_research only for the listed gaps, never
  for subtopics that are already covered. Follow the context passing rules and
  include the relevant earlier findings. Then call report_coverage again.
- If it says to finalize, stop delegating and write the final report, including a
  'Known gaps' section for anything still unresolved.
"""

In [8]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("coordinator")

# Maps each hub tool name to the spoke function that handles it
spokes = {
    "delegate_research": run_research_spoke,
    "delegate_analysis": document_analyst_spoke,
}


def coordinator(topic, max_rounds=20):
    messages = []
    add_user_message(messages, f"Research topic: {topic}")

    refinements = 0     # follow-up rounds triggered so far (separate from max_rounds)
    evaluated = False   # has the hub reported coverage at least once?

    for round_number in range(max_rounds):
        response = chat(
            messages,
            system=system_prompt + evaluate_step_prompt,
            tools=[delegate_research, delegate_analysis, report_coverage],
        )
        add_assistant_message(messages, response)
        print(f"Round {round_number + 1}: {response.stop_reason}", [block.type for block in response.content])

        if response.stop_reason != "tool_use":
            # Enforced in code: no final report before at least one coverage check
            if not evaluated:
                logger.info("Hub tried to finish without evaluating coverage; sending it back")
                add_user_message(
                    messages,
                    "You have not called report_coverage yet. Call it now, before the final report.",
                )
                continue
            return text_from_message(response)

        # Route every request: spokes for delegation, the code check for coverage
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue

            logger.info("Round %d: hub called '%s'", round_number + 1, block.name)
            try:
                if block.name == "report_coverage":
                    result = evaluate_coverage(**block.input, refinements=refinements)
                    evaluated = True
                    refinements += 1
                else:
                    result = spokes[block.name](**block.input)
                tool_results.append(
                    {"type": "tool_result", "tool_use_id": block.id, "content": result}
                )
            except Exception as e:
                logger.error("Tool '%s' failed: %s", block.name, e)
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": f"Error: {e}",
                        "is_error": True,
                    }
                )

        # Every tool_use needs a matching tool_result in the next user message
        add_user_message(messages, tool_results)

    raise RuntimeError(f"Coordinator did not finish within {max_rounds} rounds")

In [10]:
res = coordinator("renewable energy technologies")

print(f"res here: {res}")

INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 1: hub called 'delegate_research'


Round 1: tool_use ['text', 'tool_use', 'tool_use', 'tool_use', 'tool_use', 'tool_use', 'tool_use', 'tool_use', 'tool_use']


INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 1: hub called 'delegate_research'
INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 1: hub called 'delegate_research'
INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 1: hub called 'delegate_research'
INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 1: hub called 'delegate_research'
INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 1: hub called 'delegate_research'
INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 1: hub called 'delegate_research'
INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 1: hub called 'delegate_re

Round 2: tool_use ['text', 'tool_use', 'tool_use', 'tool_use']


INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 3: hub called 'delegate_analysis'


Round 3: tool_use ['text', 'tool_use', 'tool_use', 'tool_use']


INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 3: hub called 'delegate_analysis'
INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 3: hub called 'delegate_analysis'
INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 4: hub called 'report_coverage'


Round 4: tool_use ['text', 'tool_use']


INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Round 5: max_tokens ['text']
res here: Good. The coverage tool indicates I have enough material to write the final report, though significant known gaps remain. Let me now write a comprehensive, synthesized report that incorporates all findings, flags contradictions honestly, and clearly documents where uncertainties exist.

---

# RENEWABLE ENERGY TECHNOLOGIES: COMPREHENSIVE RESEARCH REPORT

## Executive Summary

Renewable energy technologies have achieved dramatic deployment growth and cost competitiveness over the past five years, fundamentally reshaping global electricity generation. As of end-2025, renewables represent the fastest-growing energy sector globally, with solar and wind leading capacity additions. This report synthesizes findings across eight key dimensions: solar, wind, hydroelectric, geothermal, biomass, energy storage, grid integration, and economics.

**Key Findings:**
- Solar and wind now represent the largest installed power capacities globally (~2,800 GW and ~1,